In [0]:
import sys
from pathlib import Path
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import AzureError

# ------------------------------------------
# Ajuste de path para importar secrets
# ------------------------------------------
sys.path.append(str(Path.cwd().parent))
from src.config.secrets import get_secret

# ------------------------------------------
# Recupera segredos
# ------------------------------------------
storage_account = get_secret("AZURE-STORAGE-ACCOUNT", "AZURE_STORAGE_ACCOUNT")
storage_key = get_secret("AZURE-STORAGE-KEY", "AZURE_STORAGE_KEY")
container_name = get_secret("AZURE-CONTAINER-BRONZE", "AZURE_CONTAINER_BRONZE")

# ------------------------------------------
# Validações robustas
# ------------------------------------------
missing = []
if not storage_account:
    missing.append("AZURE-STORAGE-ACCOUNT")
if not storage_key:
    missing.append("AZURE-STORAGE-KEY")
if not container_name:
    missing.append("AZURE-CONTAINER-BRONZE")

if missing:
    raise ValueError(f"❌ Segredos ausentes: {', '.join(missing)}")

# ------------------------------------------
# Cliente Azure Blob Storage
# ------------------------------------------
try:
    account_url = f"https://{storage_account}.blob.core.windows.net"
    blob_service_client = BlobServiceClient(
        account_url=account_url,
        credential=storage_key,
        max_single_get_size=4 * 1024 * 1024,   # otimiza downloads grandes
        max_single_put_size=4 * 1024 * 1024    # otimiza uploads grandes
    )

    # Testa acesso ao container
    container_client = blob_service_client.get_container_client(container_name)
    props = container_client.get_container_properties()

    print("✅ Conexão com Azure Storage estabelecida")
    print(f"✅ Container acessível: {container_name}")
    print(f"ℹ️ Última modificação: {props['last_modified']}")

except AzureError as e:
    raise RuntimeError(f"❌ Falha ao conectar no Azure Blob Storage: {e}")


# Função que gera um registro fictício baseado na tabela INEP Alunos
def gerar_registro():
    return {
        "ano": random.choice([2024]),
        "id_municipio": str(random.randint(1000000, 9999999)),
        "id_escola": f"E{random.randint(1000,9999)}",
        "id_aluno": f"A{random.randint(100000,999999)}",
        "proficiencia": round(random.uniform(100, 300), 2),
        "data_ingestao": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

import pandas as pd
import random
import datetime
import io

# Gerar um lote de 100 registros simulados
df = pd.DataFrame([gerar_registro() for _ in range(100)])

# Visualizar os primeiros registros
print("👀 Visualização dos dados simulados:")
display(df.head())

# Converter para Parquet em memória
parquet_buffer = io.BytesIO()
df.to_parquet(parquet_buffer, index=False, engine="pyarrow")

# Nome do arquivo com partição por data
data_particao = datetime.datetime.now().strftime("%Y/%m/%d")
blob_name = f"inep_alunos_simulado/{data_particao}/dados.parquet"

# Upload para o container bronze
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)

print(f"✅ Dados simulados gravados no bronze em {blob_name}")

import time

# Simular streaming: gerar e gravar novos dados a cada 5 segundos
for i in range(2):  # número de ciclos
    df = pd.DataFrame([gerar_registro() for _ in range(10)])  # 10 registros por ciclo
    
    # Visualizar os primeiros registros de cada lote
    print(f"👀 Lote {i+1} - preview:")
    display(df.head())
    
    parquet_buffer = io.BytesIO()
    df.to_parquet(parquet_buffer, index=False, engine="pyarrow")
    
    data_particao = datetime.datetime.now().strftime("%Y/%m/%d/%H%M%S")
    blob_name = f"inep_alunos_streaming/{data_particao}/dados.parquet"
    
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
    blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)
    
    print(f"📤 Lote {i+1} enviado para {blob_name}")
    time.sleep(5)  # espera 5 segundos antes do próximo lote

